In [ ]:
# =========================
# 🚀 FULL PIPELINE
# =========================


# -------- IMPORTS --------
import os
import torch
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# -------- CONFIG --------
class Config:
    MAX_LEN = 256
    BATCH_SIZE = 16
    EPOCHS = 2
    LR = 2e-5
    SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(Config.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("⚡ Device:", device)

# -------- LOAD DATA --------
mhqa_df = pd.read_csv("/content/mhqa_gold.csv")
mhqa_b_df = pd.read_csv("/content/mhqa_pseudo.csv")

TEXT_COLUMN = "question"
LABEL_COLUMN = "topic"

mhqa_df = mhqa_df[[TEXT_COLUMN, LABEL_COLUMN]].dropna()
mhqa_b_df = mhqa_b_df[[TEXT_COLUMN, LABEL_COLUMN]].dropna()

mhqa_df[TEXT_COLUMN] = mhqa_df[TEXT_COLUMN].str.lower().str.strip()
mhqa_b_df[TEXT_COLUMN] = mhqa_b_df[TEXT_COLUMN].str.lower().str.strip()

# -------- LABEL ENCODING (FIXED) --------
label_encoder = LabelEncoder()

# Fit on combined labels (IMPORTANT FIX)
all_labels = pd.concat([mhqa_df[LABEL_COLUMN], mhqa_b_df[LABEL_COLUMN]])
label_encoder.fit(all_labels)

mhqa_df[LABEL_COLUMN] = label_encoder.transform(mhqa_df[LABEL_COLUMN])
mhqa_b_df[LABEL_COLUMN] = label_encoder.transform(mhqa_b_df[LABEL_COLUMN])

NUM_LABELS = len(label_encoder.classes_)
print("🏷 Label Mapping:", dict(zip(label_encoder.classes_, range(NUM_LABELS))))

# -------- SPLIT --------
train_df = mhqa_b_df.copy()
val_df = mhqa_df.sample(frac=0.5, random_state=42)
test_df = mhqa_df.drop(val_df.index)

# -------- DATASET --------
class MHQADataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=Config.MAX_LEN,
            return_tensors='pt'
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# -------- MODEL --------
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
).to(device)

# -------- DATALOADERS --------
train_loader = DataLoader(
    MHQADataset(train_df[TEXT_COLUMN], train_df[LABEL_COLUMN], tokenizer),
    batch_size=Config.BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    MHQADataset(val_df[TEXT_COLUMN], val_df[LABEL_COLUMN], tokenizer),
    batch_size=Config.BATCH_SIZE
)

test_loader = DataLoader(
    MHQADataset(test_df[TEXT_COLUMN], test_df[LABEL_COLUMN], tokenizer),
    batch_size=Config.BATCH_SIZE
)

# -------- CLASS WEIGHTS --------
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df[LABEL_COLUMN]),
    y=train_df[LABEL_COLUMN]
)

class_weights = torch.tensor(weights, dtype=torch.float).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

# -------- OPTIMIZER --------
optimizer = AdamW(model.parameters(), lr=Config.LR)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=len(train_loader) * Config.EPOCHS
)

# -------- TRAIN --------
best_acc = 0

for epoch in range(Config.EPOCHS):
    print(f"\n🔥 Epoch {epoch+1}")

    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):
        optimizer.zero_grad()

        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=ids, attention_mask=mask)
        loss = loss_fn(outputs.logits, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    print("Train Loss:", total_loss / len(train_loader))

    # -------- VALIDATION --------
    model.eval()
    preds, labels_list = [], []

    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)

            outputs = model(input_ids=ids, attention_mask=mask)
            pred = torch.argmax(outputs.logits, dim=1)

            preds.extend(pred.cpu().numpy())
            labels_list.extend(batch["labels"].numpy())

    acc = accuracy_score(labels_list, preds)
    print("Validation Accuracy:", acc)

    # -------- SAVE BEST MODEL --------
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "/content/model.pt")
        print("✅ Model saved!")

# -------- TEST --------
model.eval()
preds, labels_list = [], []

with torch.no_grad():
    for batch in test_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=ids, attention_mask=mask)
        pred = torch.argmax(outputs.logits, dim=1)

        preds.extend(pred.cpu().numpy())
        labels_list.extend(batch["labels"].numpy())

print("\n📊 FINAL METRICS")
print("Accuracy:", accuracy_score(labels_list, preds))
print("F1 Score:", f1_score(labels_list, preds, average="macro"))

print("\n📄 Classification Report:")
print(classification_report(labels_list, preds))

# -------- SAVE TOKENIZER --------
tokenizer.save_pretrained("/content/tokenizer")

print("\n✅ Model + Tokenizer saved successfully")